# Πρόκληση: Ανάλυση Κειμένου για την Επιστήμη Δεδομένων

Σε αυτό το παράδειγμα, ας κάνουμε μια απλή άσκηση που καλύπτει όλα τα βήματα μιας παραδοσιακής διαδικασίας επιστήμης δεδομένων. Δεν χρειάζεται να γράψετε κώδικα, μπορείτε απλά να κάνετε κλικ στα παρακάτω κελιά για να τα εκτελέσετε και να παρατηρήσετε το αποτέλεσμα. Ως πρόκληση, ενθαρρύνεστε να δοκιμάσετε αυτόν τον κώδικα με διαφορετικά δεδομένα.

## Στόχος

Σε αυτό το μάθημα, έχουμε συζητήσει διάφορες έννοιες σχετικές με την Επιστήμη Δεδομένων. Ας προσπαθήσουμε να ανακαλύψουμε περισσότερες σχετικές έννοιες κάνοντας λίγη **εξόρυξη κειμένου**. Θα ξεκινήσουμε με ένα κείμενο για την Επιστήμη Δεδομένων, θα εξάγουμε λέξεις-κλειδιά από αυτό, και στη συνέχεια θα προσπαθήσουμε να οπτικοποιήσουμε το αποτέλεσμα.

Ως κείμενο, θα χρησιμοποιήσω τη σελίδα για την Επιστήμη Δεδομένων από τη Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Βήμα 1: Λήψη των δεδομένων

Το πρώτο βήμα σε κάθε διαδικασία επιστήμης δεδομένων είναι η λήψη των δεδομένων. Θα χρησιμοποιήσουμε τη βιβλιοθήκη `requests` για να το κάνουμε:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Βήμα 2: Μετατροπή των Δεδομένων

Το επόμενο βήμα είναι να μετατρέψουμε τα δεδομένα στη μορφή που είναι κατάλληλη για επεξεργασία. Στην περίπτωσή μας, κατεβάσαμε τον πηγαίο κώδικα HTML από τη σελίδα, και πρέπει να τον μετατρέψουμε σε απλό κείμενο.

Υπάρχουν πολλοί τρόποι για να γίνει αυτό. Θα χρησιμοποιήσουμε το [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), μια δημοφιλή βιβλιοθήκη Python για ανάλυση HTML. Το BeautifulSoup μας επιτρέπει να στοχεύσουμε συγκεκριμένα στοιχεία HTML, ώστε να μπορούμε να εστιάσουμε στο κύριο περιεχόμενο του άρθρου από τη Wikipedia και να μειώσουμε κάποια μενού πλοήγησης, πλαϊνές μπάρες, υποσέλιδα και άλλο άσχετο περιεχόμενο (αν και κάποιο επαναλαμβανόμενο κείμενο μπορεί να παραμείνει).


Πρώτα, πρέπει να εγκαταστήσουμε τη βιβλιοθήκη BeautifulSoup για την ανάλυση HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Βήμα 3: Απόκτηση Εμπειριών

Το πιο σημαντικό βήμα είναι να μετατρέψουμε τα δεδομένα μας σε κάποια μορφή από την οποία μπορούμε να εξάγουμε εμπειρίες. Στην περίπτωσή μας, θέλουμε να εξάγουμε λέξεις-κλειδιά από το κείμενο και να δούμε ποιες λέξεις-κλειδιά είναι πιο σημαντικές.

Θα χρησιμοποιήσουμε τη βιβλιοθήκη Python που ονομάζεται [RAKE](https://github.com/aneesha/RAKE) για την εξαγωγή λέξεων-κλειδιών. Αρχικά, ας εγκαταστήσουμε αυτή τη βιβλιοθήκη σε περίπτωση που δεν είναι ήδη παρούσα: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Η κύρια λειτουργικότητα είναι διαθέσιμη από το αντικείμενο `Rake`, το οποίο μπορούμε να προσαρμόσουμε χρησιμοποιώντας ορισμένες παραμέτρους. Στην περίπτωσή μας, θα ορίσουμε το ελάχιστο μήκος μιας λέξης-κλειδί σε 5 χαρακτήρες, τη ελάχιστη συχνότητα μιας λέξης-κλειδί στο έγγραφο σε 3, και τον μέγιστο αριθμό λέξεων σε μια λέξη-κλειδί σε 2. Μη διστάσετε να πειραματιστείτε με άλλες τιμές και να παρατηρήσετε το αποτέλεσμα.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Λάβαμε μια λίστα όρων μαζί με τον σχετικό βαθμό σημαντικότητας. Όπως μπορείτε να δείτε, οι πιο σχετικές επιστήμες, όπως η μηχανική μάθηση και τα μεγάλα δεδομένα, είναι παρούσες στη λίστα στις κορυφαίες θέσεις.

## Βήμα 4: Οπτικοποίηση του Αποτελέσματος

Οι άνθρωποι μπορούν να ερμηνεύσουν καλύτερα τα δεδομένα σε οπτική μορφή. Επομένως συχνά έχει νόημα να οπτικοποιήσουμε τα δεδομένα για να αντλήσουμε ορισμένες πληροφορίες. Μπορούμε να χρησιμοποιήσουμε τη βιβλιοθήκη `matplotlib` στην Python για να σχεδιάσουμε μια απλή κατανομή των λέξεων-κλειδιών με τη σχετικότητά τους:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Υπάρχει, ωστόσο, ένας ακόμη καλύτερος τρόπος να απεικονίσουμε τις συχνότητες των λέξεων - χρησιμοποιώντας **Word Cloud**. Θα χρειαστούμε να εγκαταστήσουμε μια ακόμη βιβλιοθήκη για να σχεδιάσουμε το word cloud από τη λίστα λέξεων κλειδιών μας.


In [ ]:
!{sys.executable} -m pip install wordcloud

Το αντικείμενο `WordCloud` είναι υπεύθυνο για τη λήψη είτε του αρχικού κειμένου είτε μιας προϋπολογισμένης λίστας λέξεων με τις συχνότητές τους, και επιστρέφει μια εικόνα, η οποία στη συνέχεια μπορεί να εμφανιστεί χρησιμοποιώντας το `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Μπορούμε επίσης να περάσουμε το αρχικό κείμενο στο `WordCloud` - ας δούμε αν μπορούμε να πάρουμε παρόμοιο αποτέλεσμα: 


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Μπορείτε να δείτε ότι το σύννεφο λέξεων τώρα φαίνεται πιο εντυπωσιακό, αλλά περιέχει επίσης πολύ θόρυβο (π.χ. λέξεις άσχετες όπως `Retrieved on`). Επιπλέον, έχουμε λιγότερες λέξεις-κλειδιά που αποτελούνται από δύο λέξεις, όπως *data scientist* ή *computer science*. Αυτό οφείλεται στο γεγονός ότι ο αλγόριθμος RAKE κάνει πολύ καλύτερη δουλειά στην επιλογή καλών λέξεων-κλειδιών από το κείμενο. Αυτό το παράδειγμα δείχνει τη σημασία της προεπεξεργασίας και του καθαρισμού των δεδομένων, γιατί μια καθαρή εικόνα στο τέλος θα μας επιτρέψει να πάρουμε καλύτερες αποφάσεις.

Σε αυτή την άσκηση ακολουθήσαμε μια απλή διαδικασία εξαγωγής κάποιου νοήματος από κείμενο της Wikipedia, με τη μορφή λέξεων-κλειδιών και σύννεφου λέξεων. Αυτό το παράδειγμα είναι αρκετά απλό, αλλά παρουσιάζει καλά όλα τα τυπικά βήματα που θα ακολουθήσει ένας επιστήμονας δεδομένων όταν εργάζεται με δεδομένα, ξεκινώντας από την απόκτηση δεδομένων έως την απεικόνιση.

Στο μάθημά μας θα συζητήσουμε όλα αυτά τα βήματα λεπτομερώς. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
